In [1]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import nltk
import spacy
from bs4 import BeautifulSoup
import contractions


# Gensim (LDA)
import scipy
import gensim
from gensim.corpora import Dictionary
from gensim.models import LdaModel, CoherenceModel

# Sklearn (LSA, NMF, TF-IDF)
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import TruncatedSVD, NMF
from sklearn.preprocessing import Normalizer

# Visualization
from wordcloud import WordCloud
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis

# NLTK resources
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
from nltk.corpus import stopwords

# spaCy model
try:
    nlp = spacy.load('en_core_web_sm')
except OSError:
    import subprocess
    subprocess.run(['python', '-m', 'spacy', 'download', 'en_core_web_sm'])
    nlp = spacy.load('en_core_web_sm')

print('✅ All imports successful')

✅ All imports successful


In [27]:
#Retrieve the dataset without having to import the dataset again

%store -r disneyland_df

In [29]:
disneyland_df

,Review_ID,Rating,Year_Month,Reviewer_Location,Review_Text,Branch,review_cleaned,clean_review_wo_stopwords,tokenized_texts,cleaned_texts
0,670772142,4,2019-4,Australia,If you've ever been to Disneyland anywhere you...,Disneyland_HongKong,if you have ever been to disneyland anywhere y...,ever anywhere find similar layout walk main st...,"[ever, anywhere, find, similar, layout, walk, ...",ever anywhere find similar layout walk main st...
1,670682799,4,2019-5,Philippines,Its been a while since d last time we visit HK...,Disneyland_HongKong,its been a while since d last time we visit hk...,since last visit hk yet stay tomorrowland aka ...,"[since, last, visit, yet, stay, tomorrowland, ...",since last visit yet stay tomorrowland aka mar...
2,670623270,4,2019-4,United Arab Emirates,Thanks God it wasn t too hot or too humid wh...,Disneyland_HongKong,thanks god it wasn t too hot or too humid when...,thanks god hot humid visiting otherwise would ...,"[thank, god, hot, humid, visit, otherwise, wou...",thank god hot humid visit otherwise would big ...
3,670607911,4,2019-4,Australia,HK Disneyland is a great compact park. Unfortu...,Disneyland_HongKong,hk disneyland is a great compact park unfortu...,hk great compact unfortunately quite bit maint...,"[great, compact, unfortunately, quite, bit, ma...",great compact unfortunately quite bit maintena...
4,670607296,4,2019-4,United Kingdom,"the location is not in the city, took around 1...",Disneyland_HongKong,the location is not in the city took around ...,location city took around hour kowlon kids lik...,"[location, city, take, around, hour, kowlon, k...",location city take around hour kowlon kid like...
...,...,...,...,...,...,...,...,...,...,...
42651,1765031,5,missing,United Kingdom,i went to disneyland paris in july 03 and thou...,Disneyland_Paris,i went to disneyland paris in july and thou...,july thought brilliant visited hotels stayed n...,"[july, think, brilliant, visit, hotel, stay, n...",july think brilliant visit hotel stay newport ...
42652,1659553,5,missing,Canada,2 adults and 1 child of 11 visited Disneyland ...,Disneyland_Paris,adults and child of visited disneyland ...,adults child visited beginning feb absolute fa...,"[adult, child, visit, begin, feb, absolute, fa...",adult child visit begin feb absolute fantastic...
42653,1645894,5,missing,South Africa,My eleven year old daughter and myself went to...,Disneyland_Paris,my eleven year old daughter and myself went to...,eleven year old daughter visit son london deci...,"[eleven, year, old, daughter, visit, son, lond...",eleven year old daughter visit son london deci...
42654,1618637,4,missing,United States,"This hotel, part of the Disneyland Paris compl...",Disneyland_Paris,this hotel part of the disneyland paris compl...,hotel part complex wonderful place families si...,"[hotel, part, complex, wonderful, place, famil...",hotel part complex wonderful place family sinc...


In [31]:
tokenized_texts=disneyland_df['tokenized_texts']

In [33]:
# Gensim Dictionary: maps each unique token to an integer ID
dictionary = Dictionary(tokenized_texts)

# Filter extremes: remove tokens that appear in <5 docs or >80% of docs
dictionary.filter_extremes(no_below=5, no_above=0.80)

# Bag-of-Words corpus: list of (token_id, count) per document
bow_corpus = [dictionary.doc2bow(text) for text in tokenized_texts]

print(f'Vocabulary size: {len(dictionary):,} unique tokens')
print(f'Corpus size:     {len(bow_corpus):,} documents')
print()
print('Example — first document as BoW (first 8 pairs):')
print(bow_corpus[0][:8])
print()
print('Decoded:')
print([(dictionary[wid], cnt) for wid, cnt in bow_corpus[0][:8]])

Vocabulary size: 9,098 unique tokens
Corpus size:     42,591 documents

Example — first document as BoW (first 8 pairs):
[(0, 1), (1, 1), (2, 1), (3, 1), (4, 1), (5, 2), (6, 1), (7, 1)]

Decoded:
[('absolutely', 1), ('anywhere', 1), ('busy', 1), ('ever', 1), ('fabulous', 1), ('fairly', 2), ('familiar', 1), ('feel', 1)]


In [ ]:
# ── Coherence over a range of k ─────────────────────────────────
# WARNING: this cell may take 2-5 minutes depending on machine

K_RANGE = range(2, 15)
coherence_scores = []
perplexity_scores = []

for k in K_RANGE:
    model = LdaModel(
        corpus=bow_corpus,
        id2word=dictionary,
        num_topics=k,
        random_state=42,
        passes=5,           # lower for speed during search
        chunksize=200,
        alpha='symmetric',
        eta='auto'
    )
    # Coherence (C_v) — higher is better
    cm = CoherenceModel(
        model=model, texts=tokenized_texts,
        dictionary=dictionary, coherence='c_v'
    )
    coherence_scores.append(cm.get_coherence())

    # Perplexity — lower is better
    perplexity_scores.append(np.exp(-model.log_perplexity(bow_corpus)))

print('k   Coherence  Perplexity')
print('─' * 30)
for k, c, p in zip(K_RANGE, coherence_scores, perplexity_scores):
    print(f'{k:2d}  {c:.4f}     {p:.1f}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Coherence
ax1.plot(list(K_RANGE), coherence_scores, 'o-', color='#028090', linewidth=2, markersize=7)
ax1.set_xlabel('Number of Topics (k)', fontsize=12)
ax1.set_ylabel('Coherence Score (C_v)', fontsize=12)
ax1.set_title('Coherence vs k\n(Higher = Better)', fontsize=13)
ax1.axvline(x=coherence_scores.index(max(coherence_scores)) + 2,
            color='red', linestyle='--', alpha=0.6,
            label=f'Best k = {coherence_scores.index(max(coherence_scores)) + 2}')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Perplexity
ax2.plot(list(K_RANGE), perplexity_scores, 's-', color='#1A5276', linewidth=2, markersize=7)
ax2.set_xlabel('Number of Topics (k)', fontsize=12)
ax2.set_ylabel('Perplexity', fontsize=12)
ax2.set_title('Perplexity vs k\n(Lower = Better)', fontsize=13)
ax2.grid(True, alpha=0.3)

plt.suptitle('LDA Hyperparameter Tuning — Choosing k', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

best_k = coherence_scores.index(max(coherence_scores)) + 2
print(f'Best k by coherence: {best_k}')
print(f'Max coherence score: {max(coherence_scores):.4f}')

In [ ]:
NUM_TOPICS = 4  # Change this based on elbow chart

lda_model = LdaModel(
    corpus=bow_corpus,
    id2word=dictionary,
    num_topics=NUM_TOPICS,
    random_state=42,
    passes=15,           # more passes → better convergence
    chunksize=200,
    alpha='auto',        # auto-tune doc-topic sparsity
    eta='auto'           # auto-tune topic-word sparsity
)
print(f'✅ LDA model trained with k={NUM_TOPICS}')

In [ ]:
print('=== LDA Topic-Word Distributions ===\n')
for i in range(NUM_TOPICS):
    words = lda_model.show_topic(i, topn=12)
    word_str = '  |  '.join([f'{w} ({p:.3f})' for w, p in words])
    print(f'Topic {i}: {word_str}')
    print()

In [ ]:
# ── Evaluate final model ─────────────────────────────────────────
cm_final = CoherenceModel(
    model=lda_model, texts=tokenized_texts,
    dictionary=dictionary, coherence='c_v'
)
coherence_final = cm_final.get_coherence()
perplexity_final = np.exp(-lda_model.log_perplexity(bow_corpus))

print(f'Final Coherence (C_v): {coherence_final:.4f}  (higher = better)')
print(f'Final Perplexity:      {perplexity_final:.2f}  (lower  = better)')

In [ ]:
fig, axes = plt.subplots(1, NUM_TOPICS, figsize=(5 * NUM_TOPICS, 4))
lda_topic_words = {}

for topic_idx in range(NUM_TOPICS):
    word_weights = dict(lda_model.show_topic(topic_idx, topn=30))
    lda_topic_words[f'Topic {topic_idx}'] = list(word_weights.keys())[:10]

    wc = WordCloud(
        width=600, height=400,
        background_color='white',
        colormap='Blues'
    ).generate_from_frequencies(word_weights)

    axes[topic_idx].imshow(wc, interpolation='bilinear')
    axes[topic_idx].axis('off')
    axes[topic_idx].set_title(f'Topic {topic_idx}', fontsize=14, fontweight='bold')

plt.suptitle('LDA — Top Words per Topic', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
def get_doc_topic_df(model, corpus, n_topics):
    """Return a DataFrame of topic probabilities per document."""
    rows = []
    for doc in corpus:
        dist = dict(model.get_document_topics(doc, minimum_probability=0))
        rows.append([dist.get(i, 0.0) for i in range(n_topics)])
    return pd.DataFrame(rows, columns=[f'Topic_{i}' for i in range(n_topics)])

doc_topic_df = get_doc_topic_df(lda_model, bow_corpus, NUM_TOPICS)
doc_topic_df.head(8).round(3)

In [ ]:
# Average topic proportion across corpus
avg = doc_topic_df.mean()
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(avg.index, avg.values, color='#028090', edgecolor='white', linewidth=1.5)
ax.set_ylabel('Avg Topic Proportion', fontsize=12)
ax.set_title('LDA — Average Topic Distribution Across Corpus', fontsize=13)
for bar, val in zip(bars, avg.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# What is the dominant topic for each document?
doc_topic_df['dominant_topic'] = doc_topic_df.iloc[:, :NUM_TOPICS].idxmax(axis=1)

# Sample: show a few docs and their dominant topic
sample = doc_topic_df[['dominant_topic']].copy()
sample['text_preview'] = [d[:80] + '...' for d in documents_clean[:len(sample)]]
print(sample.head(10).to_string())

print(f'\nTopic distribution across documents:')
print(doc_topic_df['dominant_topic'].value_counts())

In [ ]:
# ── pyLDAvis — run in Jupyter for interactive chart ──────────────
# Left panel:  circles = topics; size = prevalence; spacing = distinctiveness
# Right panel: word bars for selected topic
# λ slider:    0 = most unique words; 1 = most frequent; try λ = 0.6

pyLDAvis.enable_notebook()
lda_vis = gensimvis.prepare(lda_model, bow_corpus, dictionary, sort_topics=False)
pyLDAvis.display(lda_vis)

In [ ]:
# After inspecting pyLDAvis and word clouds, assign human-readable names
# ADJUST THESE based on your actual output words
TOPIC_NAMES = {
    'Topic 0': 'Space & Astronomy',
    'Topic 1': 'Baseball & Sports',
    'Topic 2': 'Gun Politics & Rights',
    'Topic 3': 'Computer Graphics & Tech',
}
print('Topic Names:')
for k, v in TOPIC_NAMES.items():
    top_words = ', '.join(lda_topic_words[k][:6])
    print(f'  {k} → {v:30s}  ({top_words})')